In [2]:
import numpy as np
import pandas as pd

In [3]:
url = "phishing_email.csv"

In [4]:
df = pd.read_csv(url)

In [5]:
df.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [6]:
# 0 → Legitimate email (normal / ham)
# 1 → Phishing or spam email (anomalous)

In [7]:
df.shape

(82486, 2)

In [8]:
df['label'].value_counts()

label
1    42891
0    39595
Name: count, dtype: int64

In [9]:
df.isna().sum()

text_combined    0
label            0
dtype: int64

In [10]:
df[df['label'] == 1]['text_combined'].head(10)

3503    link dwl g 510 802 11 g wireless pci lan adapt...
3504    indemand payperview movies sports wed 15 sep 2...
3505    want lose 19 weight try adipren hello special ...
3506    cialis xanax valium viagra low price prescript...
3507    pre order see image click indelible backscatte...
3508    site links site hello name anthony lewis seeki...
3509     htmlbody font style font size 3 difcoxw pjgfe...
3510    find sex addicts area costless dating website ...
3511    code yr 983795 hi sent email last week like co...
3512    work full time earning degree wish remove futu...
Name: text_combined, dtype: object

In [11]:
df[df['label'] == 1]['text_combined'].head(5)

3503    link dwl g 510 802 11 g wireless pci lan adapt...
3504    indemand payperview movies sports wed 15 sep 2...
3505    want lose 19 weight try adipren hello special ...
3506    cialis xanax valium viagra low price prescript...
3507    pre order see image click indelible backscatte...
Name: text_combined, dtype: object

In [12]:
df.head(10)

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0
5,hpl nom may 31 2001 see attached file hplno 53...,0
6,9760 tried get fancy address came back forward...,0
7,hpl noms february 15 2000 see attached file hp...,0
8,fw pooling contract template original message ...,0
9,hpl nom march 28 2000 see attached file hplo 3...,0


In [13]:
df_small = df[['text_combined', 'label']].dropna()
df_small.shape, df_small['label'].value_counts()


((82486, 2),
 label
 1    42891
 0    39595
 Name: count, dtype: int64)

In [14]:
df_small.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [15]:
import re

df = df_small.copy()

def clean_text(t):
    t = t.lower()
    t = re.sub(r'\s+', ' ', t)        # remove extra spaces
    t = re.sub(r'\n', ' ', t)
    return t.strip()

df["text"] = df["text_combined"].astype(str).apply(clean_text)
df["y"] = df["label"].astype(int)

df = df[["text", "y"]]
print(df.shape)
df.head()

(82486, 2)


,text,y
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [16]:
print("\n===== BASIC SANITY =====")
print("Shape:", df.shape)
print("Labels:", df["y"].unique())
print("Counts:\n", df["y"].value_counts())

print("\n===== TEXT CHECK =====")
print("Empty texts:", (df["text"].str.len() == 0).sum())
print("Min length:", df["text"].str.len().min())
print("Mean length:", df["text"].str.len().mean())
print("Max length:", df["text"].str.len().max())

print("\n===== SAMPLE TEXTS =====")
print(df["text"].sample(5, random_state=42).tolist())


===== BASIC SANITY =====
Shape: (82486, 2)
Labels: [0 1]
Counts:
 y
1    42891
0    39595
Name: count, dtype: int64

===== TEXT CHECK =====
Empty texts: 1
Min length: 0
Mean length: 1288.73811313435
Max length: 4279526

===== SAMPLE TEXTS =====
['endangered languages workshop foundation endangered languages pleased announce first workshop entitled steps language rescue take place university york weekend 26 27 july year programme saturday 2 00 2 30 arrival late registration 2 30 2 40 introduction foundation endangered languages fel committee session 2 40 3 10 endangered language policy india mahendra verma 3 10 3 40 situation berber languages north africa farid aitsiselmi 3 40 4 10 script groups use particular areas john clews session ii 4 30 5 00 izhorian estonia sweden language revival possible ilya nikolaev 5 00 5 30 issues standardisation tsimshian language american north west tonya nicole stebbins 5 30 6 00 overview endangered languages brunei darussalam peter martin break dinner 

In [17]:
print("\n===== ANOMALY SETUP CHECK =====")
print("Normal emails:", (df["y"]==0).sum())
print("Phishing emails:", (df["y"]==1).sum())
print("Phishing ratio:", df["y"].mean())


===== ANOMALY SETUP CHECK =====
Normal emails: 39595
Phishing emails: 42891
Phishing ratio: 0.5199791479766264


In [18]:
X_all = df["text"].values
y_all = df["y"].values

X_train_text = df[df["y"] == 0]["text"].values   # normal only
X_test_text = df["text"].values                  # all
y_test = df["y"].values

print("\nTrain (normal only):", len(X_train_text))
print("Test (all):", len(X_test_text))
print("Spam % in test:", y_test.mean())


Train (normal only): 39595
Test (all): 82486
Spam % in test: 0.5199791479766264


In [19]:
# Remove empty texts
df = df[df["text"].str.len() > 20]

# Remove extremely long texts
df = df[df["text"].str.len() < 5000]

df = df.reset_index(drop=True)

print("New shape:", df.shape)
print("Label counts:\n", df["y"].value_counts())
print("Mean length:", df["text"].str.len().mean())
print("Max length:", df["text"].str.len().max())

New shape: (79978, 2)
Label counts:
 y
1    42170
0    37808
Name: count, dtype: int64
Mean length: 918.5103778539099
Max length: 4997


In [20]:
!pip install sentence-transformers


In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm


In [22]:
model = SentenceTransformer("BAAI/bge-large-en-v1.5")


In [23]:
test_vec = model.encode(["hello world"])
print(test_vec.shape)


(1, 1024)


In [24]:
X_train_text = df[df["y"] == 0]["text"].tolist()   # normal only
X_test_text  = df["text"].tolist()                 # all
y_test       = df["y"].values

print("Train normal:", len(X_train_text))
print("Test total:", len(X_test_text))


Train normal: 37808
Test total: 79978


In [25]:
def embed_in_batches(texts, batch_size=64):
    vectors = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        emb = model.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        vectors.append(emb)
    return np.vstack(vectors)


In [26]:
X_train_emb = embed_in_batches(X_train_text)
X_test_emb  = embed_in_batches(X_test_text)

print("Train emb shape:", X_train_emb.shape)
print("Test emb shape:", X_test_emb.shape)


100%|██████████| 1250/1250 [4:50:10<00:00, 13.93s/it]  


Train emb shape: (37808, 1024)
Test emb shape: (79978, 1024)


In [28]:
np.save("phishing_bge_1024_train.npy", X_train_emb)
np.save("phishing_bge_1024_test.npy", X_test_emb)
np.save("phishing_bge_1024_y_test.npy", y_test)

In [29]:
# ==============================
# 384-DIM EMBEDDINGS (ONE CELL)
# Model: all-MiniLM-L6-v2 → 384 dims
# ==============================

from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

# 1) Load model (384-dim)
model_384 = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2) Sanity check
print("Embedding dim test:", model_384.encode(["hello world"]).shape)

# 3) Prepare splits
X_train_text = df[df["y"] == 0]["text"].tolist()   # normal only
X_test_text  = df["text"].tolist()                 # all
y_test       = df["y"].values

print("Train normal:", len(X_train_text))
print("Test total:", len(X_test_text))

# 4) Batch embedding function
def embed_in_batches(texts, batch_size=64):
    vectors = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        emb = model_384.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False
        )
        vectors.append(emb)
    return np.vstack(vectors)

# 5) Generate embeddings
X_train_emb_384 = embed_in_batches(X_train_text)
X_test_emb_384  = embed_in_batches(X_test_text)

# 6) Final sanity prints
print("Train emb shape:", X_train_emb_384.shape)
print("Test emb shape:", X_test_emb_384.shape)
print("Mean/std:", np.mean(X_train_emb_384), np.std(X_train_emb_384))

# 7) Save with clear names
np.save("phishing_minilm_384_train.npy", X_train_emb_384)
np.save("phishing_minilm_384_test.npy", X_test_emb_384)
np.save("phishing_minilm_384_y_test.npy", y_test)


Embedding dim test: (1, 384)
Train normal: 37808
Test total: 79978


100%|██████████| 1250/1250 [13:24<00:00,  1.55it/s]


Train emb shape: (37808, 384)
Test emb shape: (79978, 384)
Mean/std: -0.0027773904 0.0509554


In [30]:
# ===== Sanity checks for both embeddings (1024 + 384) =====
import numpy as np

# Load
Xtr_1024 = np.load("phishing_bge_1024_train.npy")
Xte_1024 = np.load("phishing_bge_1024_test.npy")
yte_1024 = np.load("phishing_bge_1024_y_test.npy")

Xtr_384 = np.load("phishing_minilm_384_train.npy")
Xte_384 = np.load("phishing_minilm_384_test.npy")
yte_384 = np.load("phishing_minilm_384_y_test.npy")

def check_embeddings(name, Xtr, Xte, yte, expected_dim):
    print(f"\n===== {name} SANITY =====")
    # Shapes
    print("Train shape:", Xtr.shape)
    print("Test  shape:", Xte.shape)
    print("y_test shape:", yte.shape)
    assert Xtr.shape[1] == expected_dim, f"{name}: train dim != {expected_dim}"
    assert Xte.shape[1] == expected_dim, f"{name}: test dim != {expected_dim}"
    assert Xte.shape[0] == yte.shape[0], f"{name}: test rows != y_test length"

    # Dtypes
    print("Train dtype:", Xtr.dtype, " | Test dtype:", Xte.dtype)

    # NaN / Inf
    print("NaNs (train/test):", np.isnan(Xtr).sum(), "/", np.isnan(Xte).sum())
    print("Infs (train/test):", np.isinf(Xtr).sum(), "/", np.isinf(Xte).sum())

    # Basic stats
    print("Mean/std (train):", float(Xtr.mean()), float(Xtr.std()))
    print("Mean/std (test) :", float(Xte.mean()), float(Xte.std()))

    # Norm check (since normalize_embeddings=True)
    tr_norms = np.linalg.norm(Xtr, axis=1)
    te_norms = np.linalg.norm(Xte, axis=1)
    print("Norms train (min/mean/max):", tr_norms.min(), tr_norms.mean(), tr_norms.max())
    print("Norms test  (min/mean/max):", te_norms.min(), te_norms.mean(), te_norms.max())

    # Quick sample vectors
    print("Sample train[0][:5]:", Xtr[0][:5])
    print("Sample test[0][:5] :", Xte[0][:5])

# Run checks
check_embeddings("BGE-1024", Xtr_1024, Xte_1024, yte_1024, expected_dim=1024)
check_embeddings("MiniLM-384", Xtr_384, Xte_384, yte_384, expected_dim=384)


===== BGE-1024 SANITY =====
Train shape: (37808, 1024)
Test  shape: (79978, 1024)
y_test shape: (79978,)
Train dtype: float32  | Test dtype: float32
NaNs (train/test): 0 / 0
Infs (train/test): 0 / 0
Mean/std (train): -0.00021349990856833756 0.03124985657632351
Mean/std (test) : -0.00020924827549606562 0.03124993108212948
Norms train (min/mean/max): 0.9999999 1.0 1.0000001
Norms test  (min/mean/max): 0.9999999 1.0 1.0000001
Sample train[0][:5]: [ 0.03681918 -0.01246196 -0.00417001  0.05225328 -0.04238624]
Sample test[0][:5] : [ 0.03681918 -0.01246196 -0.00417001  0.05225328 -0.04238624]

===== MiniLM-384 SANITY =====
Train shape: (37808, 384)
Test  shape: (79978, 384)
y_test shape: (79978,)
Train dtype: float32  | Test dtype: float32
NaNs (train/test): 0 / 0
Infs (train/test): 0 / 0
Mean/std (train): -0.0027773904148489237 0.0509553998708725
Mean/std (test) : -0.0027576617430895567 0.05095649138092995
Norms train (min/mean/max): 0.9999999 1.0 1.0000001
Norms test  (min/mean/max): 0.999